In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("data.xlsx", sheet_name="in")
print("Original shape:", df.shape)

Original shape: (5986, 10)


In [2]:
n_full_dupes = df.duplicated().sum()
print("Fully duplicated rows:", n_full_dupes)

df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing exact duplicates:", df.shape)

Fully duplicated rows: 0
Shape after removing exact duplicates: (5986, 10)


In [3]:
name_norm = df["name"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
dupe_mask = name_norm.duplicated(keep="first")
print("Case/whitespace duplicate names:", dupe_mask.sum())

df = df.loc[~dupe_mask].reset_index(drop=True)
print("Shape after removing soft duplicates:", df.shape)

Case/whitespace duplicate names: 0
Shape after removing soft duplicates: (5986, 10)


In [4]:
text_cols = ["name", "planet_status"]
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

df[text_cols].dtypes

name             str
planet_status    str
dtype: object

In [5]:
numeric_cols = ["mass", "mass_error_min", "mass_error_max",
                 "mass_sini", "mass_sini_error_min", "mass_sini_error_max",
                 "radius", "radius_error_min"]

for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[numeric_cols].dtypes

mass                   float64
mass_error_min         float64
mass_error_max         float64
mass_sini              float64
mass_sini_error_min    float64
mass_sini_error_max    float64
radius                 float64
radius_error_min       float64
dtype: object

In [6]:
df["planet_status"] = df["planet_status"].astype("category")
df["planet_status"].dtype

CategoricalDtype(categories=['Confirmed'], ordered=False, categories_dtype=str)

In [7]:
inf_mask = np.isinf(df[numeric_cols])
print("Infinite values found:", inf_mask.sum().sum())

df[numeric_cols] = df[numeric_cols].mask(inf_mask, np.nan)

Infinite values found: 9


In [8]:
zero_cols = ["mass", "mass_sini", "radius"]
zero_mask = df[zero_cols] == 0
print("Zero values found:", zero_mask.sum().sum())

df[zero_cols] = df[zero_cols].mask(zero_mask, np.nan)

Zero values found: 0


In [9]:
print("Final shape:", df.shape)
df.to_csv("cleaned_step_output.csv", index=False)

Final shape: (5986, 10)
